# HLS Surface Reflectance Acceptance Test

## Purpose
This notebook confirms that a new HLS container produces surface reflectance (SR) and vegetation index (VI) outputs that are numerically equivalent to a **reference container** (prod).

It compares all `.tif` band files between two S3 prefixes (e.g., `prod` vs `dev`) and determines whether observed numerical differences are statistically significant or consistent with random floating-point noise.

## Outline
1. **Configuration** — S3 paths, product types, and output directories
2. **Setup** — imports, AWS credentials, S3 client initialisation
3. **Data Discovery** — list and pair files between reference and candidate containers
4. **File Inventory Check** — verify file counts and sizes match across granules
5. **Helper Functions** — S3 file reading, twin-path resolution, stretch utilities
6. **Pixel-level Comparison** — compute per-file difference metrics and export comparison figures
7. **Statistical Testing & Pass / Fail Decision** — per-band significance tests and final verdict

## How to Run
1. Edit **Section 1** — set S3 paths for the containers to compare
2. Run **Section 2** — paste temporary AWS credentials (expires every 12 h)
3. Run **Sections 3 – 7** in order

## HLS Scaling Convention
HLS stores surface reflectance and vegetation indices as scaled integers:  
**Physical value = DN × 0.0001** (HLS V2.0 User Guide, LP DAAC)

| Product | Valid DN range | Physical range |
|---|---|---|
| SR bands (B01–B12, B8A) | −2 000 – 16 000 | −0.2 – 1.6 reflectance |
| VI products (NDVI, EVI, …) | −10 000 – 10 000 | −1.0 – 1.0 index units |

All difference metrics are reported in **physical reflectance / index units** (DN × 0.0001), not raw DN, to ensure scale-independent comparisons.


---
## 1. Configuration
Edit the variables in this cell only. Do not hardcode paths elsewhere in the notebook.

In [131]:
# ── S3 paths ──────────────────────────────────────────────────────────────────
# Include bucket name as the first component, e.g.:
#   'my-bucket/outputs/exp1/prod/'
folder_pre  = 'hls-science-container-testing-018923174646-us-west-2-an/outputs/exp1-recompile/prod/'
folder_post = 'hls-science-container-testing-018923174646-us-west-2-an/outputs/exp1-recompile/dev/'

# ── AWS region ────────────────────────────────────────────────────────────────
AWS_REGION = 'us-west-2'

# ── Product types to include in the comparison ───────────────────────────────
PRODUCT_TYPES = ['L30', 'S30', 'L30_VI', 'S30_VI']

# ── Date filter: set a Julian day (e.g. '2026059') or '' for all dates ───────
DATES = {
    'L30'    : '',
    'S30'    : '',
    'L30_VI' : '',
    'S30_VI' : '',
}

# ── Pass/Fail thresholds ──────────────────────────────────────────────────────
# All absolute difference thresholds are in physical reflectance / index units
# (DN * HLS_SCALE_FACTOR). Set a threshold to None to skip that check.
# TODO: update values based on your baseline experiment and statistical analysis (Section 9).
THRESHOLDS = {
    # Standard SR bands (B01–B09, B8A, angle bands)
    'sr_max_pct_pixels_changed'  : None,    # TODO: e.g. 0.1  (%)
    'sr_max_abs_diff_refl'       : None,    # TODO: e.g. 0.0005 reflectance units
    'sr_max_pct_of_range'        : None,    # TODO: e.g. 0.01 (%)
    # Vegetation index products (EVI, MSAVI, NBR, NDMI, NDVI, NDWI, SAVI, TVI)
    'vi_max_pct_pixels_changed'  : None,    # TODO
    'vi_max_abs_diff_refl'       : None,    # TODO: e.g. 0.005 index units
    'vi_max_pct_of_range'        : None,    # TODO
    # File inventory
    'require_identical_filesizes': True,
}

# ── Image export ─────────────────────────────────────────────────────────────
# Set True to save a comparison figure (histograms + difference map) for every
# file that has non-zero pixel differences during the comparison loop.
SAVE_COMPARISON_IMAGES = True

# ── Output directories ───────────────────────────────────────────────────────
import os
# os.path.abspath('') == CWD == notebooks/ when run from there;
# join with '..' steps into hls_validation_framework/, then into reports/ / outputs/
REPORT_DIR  = os.path.abspath(os.path.join('', '..', 'reports'))
OUTPUTS_DIR = os.path.abspath(os.path.join('', '..', 'outputs'))
os.makedirs(REPORT_DIR,  exist_ok=True)
os.makedirs(OUTPUTS_DIR, exist_ok=True)
print(f"Report  dir : {os.path.abspath(REPORT_DIR)}")
print(f"Outputs dir : {os.path.abspath(OUTPUTS_DIR)}")

Report  dir : /Users/vo25-mbp/hls-validation/hls_validation_framework/reports
Outputs dir : /Users/vo25-mbp/hls-validation/hls_validation_framework/outputs


---
## 2. Setup
Imports, module loading, and S3 client initialisation.  
**Run after** Section 1 (Configuration). Paste your AWS credentials before running this cell.

In [ ]:
import os
import tempfile
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import boto3
import rasterio
from collections import defaultdict
from botocore.exceptions import NoCredentialsError, ClientError

%run -i ../module/plotting.ipynb
%run -i ../module/data_access.ipynb
%run -i ../module/ultilities.ipynb
%run -i ../module/fmask.ipynb

# ── AWS credentials ───────────────────────────────────────────────────────────
# Set before running this notebook:
#   export AWS_ACCESS_KEY_ID=...
#   export AWS_SECRET_ACCESS_KEY=...
#   export AWS_SESSION_TOKEN=...   (if using temporary credentials)
try:
    session  = boto3.Session(region_name=AWS_REGION)
    creds    = session.get_credentials()
    assert creds is not None, "No AWS credentials found in environment."
    resolved = creds.get_frozen_credentials()
    os.environ['AWS_DEFAULT_REGION'] = AWS_REGION
    s3 = session.client('s3', region_name=AWS_REGION)
    print(f"✅ AWS credentials loaded (key: {resolved.access_key[:8]}...)")
except (NoCredentialsError, AssertionError) as e:
    print("❌ No AWS credentials found. Set environment variables:")
    print("     export AWS_ACCESS_KEY_ID=...")
    print("     export AWS_SECRET_ACCESS_KEY=...")
    print("     export AWS_SESSION_TOKEN=...")
    raise

# ── Derived bucket / prefix names ────────────────────────────────────────────
BUCKET_NAME_prod = folder_pre.split('/')[0]
BUCKET_NAME_dev  = folder_post.split('/')[0]
PREFIX_pre       = '/'.join(folder_pre.split('/')[1:])
PREFIX_post      = '/'.join(folder_post.split('/')[1:])

for _bkt in [BUCKET_NAME_prod, BUCKET_NAME_dev]:
    try:
        s3.list_objects_v2(Bucket=_bkt, MaxKeys=1)
        print(f'✅ Bucket access confirmed: s3://{_bkt}')
    except ClientError as _e:
        code = _e.response['Error']['Code']
        if code in ('403', 'AccessDenied'):
            print(f'⚠️  Bucket-level list denied for s3://{_bkt} — object-level access may still work')
        else:
            raise

---
## 3. Data Discovery
List all `.tif` files in both the reference (prod) and candidate (dev) S3 prefixes and pair them by path.

In [ ]:
def list_s3_files(bucket, prefix='', max_display=20):
    """List all objects in an S3 bucket under a given prefix."""
    paginator  = s3.get_paginator('list_objects_v2')
    files      = []
    total_size = 0
    for page in paginator.paginate(Bucket=bucket, Prefix=prefix):
        for obj in page.get('Contents', []):
            if not obj['Key'].endswith('/'):
                files.append(obj['Key'])
                total_size += obj['Size']
    print(f"\n📁 s3://{bucket}/{prefix}")
    print(f"   Total objects : {len(files)}")
    print(f"   Total size    : {total_size / (1024**3):.2f} GB")
    print(f"\n   First {min(max_display, len(files))} files:")
    for f in files[:max_display]:
        print(f"     {f}")
    if len(files) > max_display:
        print(f"     ... and {len(files) - max_display} more")
    return files


files_s3_all = {}

print("=" * 60)
print("CANDIDATE (dev)")
print("=" * 60)
for pt in PRODUCT_TYPES:
    date   = DATES.get(pt, '')
    prefix = f"{PREFIX_post}{pt}/data/{date}/" if date else f"{PREFIX_post}{pt}/data/"
    print(f"\n🔍 Listing {pt} (date: {date or 'all'}) ...")
    files_s3_all[pt] = list_s3_files(BUCKET_NAME_dev, prefix=prefix)
files_s3_dev = [f for files in files_s3_all.values() for f in files]

print("\n" + "=" * 60)
print("REFERENCE (prod)")
print("=" * 60)
for pt in PRODUCT_TYPES:
    date   = DATES.get(pt, '')
    prefix = f"{PREFIX_pre}{pt}/data/{date}/" if date else f"{PREFIX_pre}{pt}/data/"
    print(f"\n🔍 Listing {pt} (date: {date or 'all'}) ...")
    files_s3_all[pt] = list_s3_files(BUCKET_NAME_prod, prefix=prefix)
files_s3_prod = [f for files in files_s3_all.values() for f in files]

print(f"\n{'='*60}")
print(f"Total files  — prod: {len(files_s3_prod)}  |  dev: {len(files_s3_dev)}")

---
## 4. File Inventory Check
Verify that file counts per granule match between prod and dev, and detect any twin-granule structure.

In [ ]:
def granule_key(s3_key):
    """Return (granule_id, is_twin) from an S3 key."""
    parts = s3_key.split('/')
    if len(parts) > 6:
        granule = parts[6]
        is_twin = len(parts) > 7 and parts[7] == 'twin'
        return granule, is_twin
    return None, False


def file_counts(file_list):
    """Count .tif files per (granule, direct/twin) subfolder."""
    counts = defaultdict(lambda: defaultdict(list))
    for f in file_list:
        if not f.endswith('.tif'):
            continue
        granule, is_twin = granule_key(f)
        if granule:
            counts[granule]['twin' if is_twin else 'direct'].append(os.path.basename(f))
    return counts


prod_counts   = file_counts(files_s3_prod)
dev_counts    = file_counts(files_s3_dev)
twin_granules = {g for g, v in prod_counts.items() if 'twin' in v}

print(f"Twin granules in prod: {len(twin_granules)}")
print(f"\n{'Granule':<55} {'Prod direct':>11} {'Prod twin':>9} {'Dev direct':>10} {'Dev twin':>8}  {'Match?'}")
print("-" * 110)

mismatches = []
for g in sorted(twin_granules):
    p_direct = len(prod_counts[g].get('direct', []))
    p_twin   = len(prod_counts[g].get('twin',   []))
    d_direct = len(dev_counts[g].get('direct',  []))
    d_twin   = len(dev_counts[g].get('twin',    []))
    match    = '✅' if (p_direct == d_direct and p_twin == d_twin) else '❌'
    if p_direct != d_direct or p_twin != d_twin:
        mismatches.append(g)
    print(f"{g:<55} {p_direct:>11} {p_twin:>9} {d_direct:>10} {d_twin:>8}  {match}")

print(f"\n✅ Matched : {len(twin_granules) - len(mismatches)}")
print(f"❌ Mismatch: {len(mismatches)}")

# ── Remove mismatched paths so the comparison remains symmetric ───────────────
if mismatches:
    drop_direct = set()
    drop_twin   = set()
    for g in mismatches:
        if len(prod_counts[g].get('direct', [])) != len(dev_counts[g].get('direct', [])):
            drop_direct.add(g)
        if len(prod_counts[g].get('twin', [])) != len(dev_counts[g].get('twin', [])):
            drop_twin.add(g)

    def _keep(f):
        granule, is_twin = granule_key(f)
        if is_twin   and granule in drop_twin:   return False
        if not is_twin and granule in drop_direct: return False
        return True

    files_s3_prod = [f for f in files_s3_prod if _keep(f)]
    files_s3_dev  = [f for f in files_s3_dev  if _keep(f)]
    print(f"\nAfter removing mismatched paths — prod: {len(files_s3_prod)}  dev: {len(files_s3_dev)}")
else:
    print("\n✅ No mismatches — file lists unchanged")

files_s3_prod_selected = files_s3_prod

---
## 5. Helper Functions
S3 file reading, twin-path resolution, and stretch utilities.

In [ ]:
def _resolve_s3_key(bucket, s3_key):
    """Return the actual S3 key, trying a twin/ subfolder as fallback for dev buckets."""
    if 'dev' not in bucket and 'dev' not in s3_key:
        return s3_key
    try:
        s3.head_object(Bucket=bucket, Key=s3_key)
        return s3_key
    except ClientError as e:
        if e.response['Error']['Code'] not in ('404', 'NoSuchKey'):
            raise
    parts    = s3_key.rsplit('/', 1)
    twin_key = parts[0] + '/twin/' + parts[1]
    try:
        s3.head_object(Bucket=bucket, Key=twin_key)
        return twin_key
    except ClientError:
        pass
    return s3_key


def open_s3_raster(bucket_name, s3_key, band=1):
    """Stream an S3 raster into memory via boto3 and return (numpy array, s3_url)."""
    _s3      = boto3.Session(region_name=AWS_REGION).client('s3')
    tmp_path = None
    try:
        resolved_key = _resolve_s3_key(bucket_name, s3_key)
        with tempfile.NamedTemporaryFile(suffix='.tif', delete=False) as tmp:
            tmp_path = tmp.name
            response = _s3.get_object(Bucket=bucket_name, Key=resolved_key)
            for chunk in response['Body'].iter_chunks(chunk_size=8 * 1024 * 1024):
                tmp.write(chunk)
        s3_url = f"s3://{bucket_name}/{resolved_key}"
        with rasterio.open(tmp_path) as src:
            data = src.read(band).astype(np.float32)
        return data, s3_url
    except Exception as e:
        print(f"❌ Error reading {s3_key}: {e}")
        return None, None
    finally:
        if tmp_path and os.path.exists(tmp_path):
            os.remove(tmp_path)


def stretch(arr, low=2, high=98):
    """Percentile-stretch an array to [0, 1] for display."""
    lo, hi = np.nanpercentile(arr, (low, high))
    return np.clip((arr - lo) / (hi - lo + 1e-9), 0, 1)


print("✅ Helper functions defined")

---
## 6. Pixel-level Comparison
For every paired `.tif` file, load both versions, compute pixel differences, and collect metrics.

In [ ]:
import matplotlib
matplotlib.use('Agg')   # non-interactive backend; safe for both inline and script use
import matplotlib.pyplot as plt

df_diff_list = []

def _stats_label(arr):
    v = arr.flatten()
    v = v[~np.isnan(v)]
    if v.size == 0:
        return 'No valid pixels'
    return f'Min: {np.min(v):.4f}\nMax: {np.max(v):.4f}\nMean: {np.mean(v):.4f}'

def _save_comparison_figure(file, data_prod, data_dev, data_diff, _n_nonzero, _pct_nonzero):
    """Save a 2x3 comparison figure (histograms + difference map) for one file."""
    _fname_stem = file.split('/')[-1].replace('.tif', '')
    _save_path  = os.path.join(OUTPUTS_DIR, f'{_fname_stem}_comparison.png')

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    fig.suptitle(file.split('/')[-1], fontsize=11)

    # Row 1 — histograms with distribution-fitted x-axis range
    # Use mean ± 3 SD clipped to the 1st–99th percentile of the combined
    # distribution so the axis shows the bulk of the data, not extreme outliers.
    _v_prod = data_prod.flatten(); _v_prod = _v_prod[~np.isnan(_v_prod)]
    _v_dev  = data_dev.flatten();  _v_dev  = _v_dev[~np.isnan(_v_dev)]
    _v_all  = np.concatenate([_v_prod, _v_dev])
    _xmin_h = float(np.percentile(_v_all,  5))  # 90% CI lower bound
    _xmax_h = float(np.percentile(_v_all, 95))  # 90% CI upper bound
    _bins_h    = np.linspace(_xmin_h, _xmax_h, 51)  # shared bin edges

    axes[0, 0].hist(_v_prod, bins=_bins_h, color='steelblue', alpha=0.85)
    axes[0, 0].set_title('Reference (prod)')
    axes[0, 0].text(0.02, 0.97, _stats_label(data_prod), transform=axes[0, 0].transAxes,
                    va='top', fontsize=9, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
    axes[0, 0].set_xlabel('DN  (90% CI: p5–p95)', fontsize=7)

    axes[0, 1].hist(_v_dev, bins=_bins_h, color='tomato', alpha=0.85)
    axes[0, 1].set_title('Candidate (dev)')
    axes[0, 1].text(0.02, 0.97, _stats_label(data_dev), transform=axes[0, 1].transAxes,
                    va='top', fontsize=9, bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.7))
    axes[0, 1].set_xlabel('DN  (90% CI: p5–p95)', fontsize=7)

    # Shared y-axis (count) and x-axis across both histograms
    _ymax_h = max(axes[0, 0].get_ylim()[1], axes[0, 1].get_ylim()[1])
    for _ax in [axes[0, 0], axes[0, 1]]:
        _ax.set_xlim(_xmin_h, _xmax_h)
        _ax.set_ylim(0, _ymax_h)

    # Density hexbin (all pixels) + outlier overlay
    # With only ~0.08% of pixels differing, a plain scatter hides outliers.
    # Strategy: draw a hexbin density map for the bulk, then overlay the
    # non-zero-diff pixels as red markers so they are always visible.
    _p_flat  = data_prod.flatten()
    _d_flat  = data_dev.flatten()
    _df_flat = data_diff.flatten()
    _valid2  = ~np.isnan(_p_flat) & ~np.isnan(_d_flat)
    _x_all   = _p_flat[_valid2]     # reference (prod) in DN
    _y_all   = _d_flat[_valid2]     # candidate (dev) in DN
    _d_all   = _df_flat[_valid2]    # difference in DN
    _n_scatter  = int(_valid2.sum())
    _outlier_mask = _d_all != 0
    _n_nonzero2   = int(_outlier_mask.sum())
    _pct_diff     = 100.0 * _n_nonzero2 / _n_scatter if _n_scatter else 0.0
    _mean_diff    = float(np.mean(_d_all))
    # R² and significance stars
    from scipy.stats import linregress
    _slope, _intercept, _r, _pval, _ = linregress(_x_all, _y_all)
    _r2 = _r ** 2
    if   _pval < 0.001: _sig = '***'
    elif _pval < 0.01:  _sig = '**'
    elif _pval < 0.05:  _sig = '*'
    else:               _sig = 'ns'
    # Scatter: prod (x) vs dev (y) — axis range fitted to bulk distribution
    # Use the same mean ± 3 SD / p1–p99 clipping as the histograms.
    _xy_all = np.concatenate([_x_all, _y_all])
    _s_lo = float(np.percentile(_xy_all,  5))  # 90% CI lower bound
    _s_hi = float(np.percentile(_xy_all, 95))  # 90% CI upper bound
    _view = [_s_lo, _s_hi]          # axis view range (90% CI)
    _full = [min(_x_all.min(), _y_all.min()), max(_x_all.max(), _y_all.max())]  # full range for 1:1 line
    # Sub-sample up to 5 000 pixels
    _s_idx = np.random.default_rng(0).choice(_n_scatter, size=min(5_000, _n_scatter), replace=False)
    axes[0, 2].scatter(_x_all[_s_idx], _y_all[_s_idx],
                       s=50, alpha=0.6, color='steelblue', marker='o',
                       linewidths=0, zorder=2)
    axes[0, 2].plot(_full, _full, color='black', linewidth=1.5, linestyle='--',
                    label='1:1', zorder=3)
    axes[0, 2].set_xlim(_view)
    axes[0, 2].set_ylim(_view)
    axes[0, 2].set_aspect('equal', adjustable='box')
    axes[0, 2].set_xlabel('Reference / prod (DN)', fontsize=8)
    axes[0, 2].set_ylabel('Candidate / dev (DN)', fontsize=8)
    axes[0, 2].set_title('Old vs. New Container')
    axes[0, 2].text(0.04, 0.95,
                    f'R\u00b2 = {_r2:.6f}  {_sig}\n'
                    f'Mean diff = {_mean_diff:.4f} DN\n'
                    f'Pixels w/ diff = {_pct_diff:.4f}%\n'
                    f'n = {_n_scatter:,}',
                    transform=axes[0, 2].transAxes, va='top', fontsize=9,
                    bbox=dict(boxstyle='round,pad=0.3', fc='white', alpha=0.8))

    # Row 2 — spatial maps (NaN rendered as black)
    _cmap_gray = plt.cm.gray.copy()
    _cmap_gray.set_bad(color='mintcream')
    axes[1, 0].imshow(stretch(data_prod), cmap=_cmap_gray)
    axes[1, 0].set_title('Reference (stretched)')
    axes[1, 0].axis('off')

    axes[1, 1].imshow(stretch(data_dev), cmap=_cmap_gray)
    axes[1, 1].set_title('Candidate (stretched)')
    axes[1, 1].axis('off')

    # Difference map: clip colorbar to valid data range (p2–p98) to avoid
    # the colourbar being dominated by extreme single-pixel outliers.
    _abs_max  = max(float(np.nanpercentile(np.abs(data_diff), 98)), 1e-6)
    _norm     = plt.matplotlib.colors.TwoSlopeNorm(vmin=-_abs_max, vcenter=0, vmax=_abs_max)
    _cmap_div = plt.cm.RdBu_r.copy()
    _cmap_div.set_bad(color='mintcream')
    _im       = axes[1, 2].imshow(data_diff, cmap=_cmap_div, norm=_norm)
    axes[1, 2].set_title('Difference map')
    axes[1, 2].set_xlabel(f'Non-zero pixels: {_n_nonzero:,} ({_pct_nonzero:.4f}%)', fontsize=9)
    axes[1, 2].axis('off')
    plt.colorbar(_im, ax=axes[1, 2], fraction=0.046, pad=0.04)

    plt.tight_layout()
    plt.savefig(_save_path, bbox_inches='tight', dpi=100)
    plt.close(fig)
    return _save_path


for file in files_s3_prod_selected[:10]:
    if not file.endswith('.tif'):
        print(f"Skipping non-raster file: {file}")
        continue

    try:
        data_prod, filename_prod = open_s3_raster(BUCKET_NAME_prod, file)
        data_dev,  filename_dev  = open_s3_raster(BUCKET_NAME_dev,  file.replace('prod', 'dev'))

        if data_prod is None or data_dev is None:
            print(f'  Skipping {file} - one or both files failed to load')
            continue

        print(f'  PROD : {filename_prod}')
        print(f'  DEV  : {filename_dev}')

        # ── Mask fill values ─────────────────────────────────────────────────
        _FILL     = -19999 if '_VI' in file else -9999
        data_prod = np.where(data_prod == _FILL, np.nan, data_prod)
        data_dev  = np.where(data_dev  == _FILL, np.nan, data_dev)
        data_diff = np.where(
            np.isnan(data_prod) | np.isnan(data_dev), np.nan,
            data_prod - data_dev
        )

        # ── Pixel difference metrics ──────────────────────────────────────────
        _diff_flat   = data_diff.flatten()
        _valid_mask  = ~np.isnan(_diff_flat)
        _n_total     = int(_diff_flat.size)
        _n_valid     = int(_valid_mask.sum())
        _n_nonzero   = int(np.count_nonzero(_diff_flat[_valid_mask]))
        _pct_nonzero = round(100.0 * _n_nonzero / _n_total, 5) if _n_total else 0.0

        # ── Convert DN differences to physical units ─────────────────────────
        # HLS scale factor: physical_value = DN * 0.0001
        _scale          = HLS_SCALE_FACTOR
        _max_abs_diff   = float(np.nanmax(np.abs(data_diff)))
        _max_abs_diff_r = _max_abs_diff * _scale
        _rmse_dn        = float(np.sqrt(np.nanmean(data_diff ** 2)))
        _rmse_refl      = _rmse_dn * _scale
        _mean_diff_r    = float(np.nanmean(data_diff)) * _scale

        # Signal range in physical units
        _prod_refl      = data_prod * _scale
        _signal_range_r = float(np.nanmax(_prod_refl)) - float(np.nanmin(_prod_refl))
        _mean_refl      = float(np.nanmean(_prod_refl))
        _std_refl       = float(np.nanstd(_prod_refl))

        _pct_of_range  = round(100.0 * _max_abs_diff_r / _signal_range_r, 5) if _signal_range_r else float('nan')
        _pct_of_signal = round(100.0 * _max_abs_diff_r / abs(_mean_refl),  5) if _mean_refl      else float('nan')
        _pct_of_std    = round(100.0 * _max_abs_diff_r / _std_refl,        5) if _std_refl       else float('nan')

        # ── File size comparison ──────────────────────────────────────────────
        _post_key       = _resolve_s3_key(BUCKET_NAME_dev, file.replace(folder_pre, folder_post))
        _filesize_prod  = s3.head_object(Bucket=BUCKET_NAME_prod, Key=file)['ContentLength']
        _filesize_dev   = s3.head_object(Bucket=BUCKET_NAME_dev,  Key=_post_key)['ContentLength']
        _size_identical = (_filesize_prod == _filesize_dev)

        # ── Export comparison figure (only for files with differences) ────────
        #_fig_path = None
        if SAVE_COMPARISON_IMAGES:
            _fig_path = _save_comparison_figure(
                file, data_prod, data_dev, data_diff, _n_nonzero, _pct_nonzero
            )
            print(f'    Saved figure: {os.path.basename(_fig_path)}')

        df_diff = pd.DataFrame({
            'file'                  : [file],
            'product_type'          : ['VI' if '_VI' in file else 'SR'],
            'n_total_pixels'        : [_n_total],
            'n_valid_pixels'        : [_n_valid],
            'n_nonzero_diff'        : [_n_nonzero],
            'pct_nonzero_diff'      : [_pct_nonzero],
            # ── Physical-unit metrics (reflectance or index units) ──────────
            'max_abs_diff_refl'     : [round(_max_abs_diff_r, 6)],
            'rmse_refl'             : [round(_rmse_refl,      6)],
            'mean_diff_refl'        : [round(_mean_diff_r,    6)],
            'pct_of_range'          : [round(_pct_of_range,   5) if not np.isnan(_pct_of_range)  else float('nan')],
            'pct_of_signal'         : [round(_pct_of_signal,  5) if not np.isnan(_pct_of_signal) else float('nan')],
            'pct_of_std'            : [round(_pct_of_std,     5) if not np.isnan(_pct_of_std)    else float('nan')],
            # ── Summary stats in physical units ─────────────────────────────
            'max_pre_refl'          : [round(float(np.nanmax(data_prod))  * _scale, 6)],
            'max_post_refl'         : [round(float(np.nanmax(data_dev))   * _scale, 6)],
            'min_pre_refl'          : [round(float(np.nanmin(data_prod))  * _scale, 6)],
            'min_post_refl'         : [round(float(np.nanmin(data_dev))   * _scale, 6)],
            'mean_pre_refl'         : [round(float(np.nanmean(data_prod)) * _scale, 6)],
            'mean_post_refl'        : [round(float(np.nanmean(data_dev))  * _scale, 6)],
            # ── File inventory ───────────────────────────────────────────────
            'filesize_pre'          : [_filesize_prod],
            'filesize_post'         : [_filesize_dev],
            'filesize_identical'    : [_size_identical],
            'twin_granule'          : [file.split('/')[6] in twin_granules if len(file.split('/')) > 6 else False],
            'filename_pre'          : [filename_prod],
            'filename_post'         : [filename_dev],
            'figure_path'           : [_fig_path],
            'unique_diffs_dn'       : [sorted(np.unique(_diff_flat[_valid_mask]).tolist())],
        })

        df_diff_list.append(df_diff)

    except Exception as e:
        import traceback
        print(f"Error processing {file}: {e}")
        traceback.print_exc()
        if df_diff_list:
            _ts    = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
            _fpath = os.path.join(REPORT_DIR, f'sr_acceptance_checkpoint_{_ts}.csv')
            pd.concat(df_diff_list, ignore_index=True).to_csv(_fpath, index=False)
            print(f"  Progress saved ({len(df_diff_list)} files) -> {_fpath}")

df_results = pd.concat(df_diff_list, ignore_index=True)
_n_with_figs = int((df_results['figure_path'].notna()).sum())
print(f"\nComparison complete - {len(df_results)} files processed, {_n_with_figs} comparison figures saved")
df_results[['file', 'product_type', 'pct_nonzero_diff', 'max_abs_diff_refl', 'pct_of_range', 'filesize_identical', 'figure_path']].head(10)


  PROD : s3://hls-science-container-testing-018923174646-us-west-2-an/outputs/exp1-recompile/prod/S30/data/2026059/HLS.S30.T11RPM.2026059T182209.v2.0/HLS.S30.T11RPM.2026059T182209.v2.0.B02.tif
  DEV  : s3://hls-science-container-testing-018923174646-us-west-2-an/outputs/exp1-recompile/dev/S30/data/2026059/HLS.S30.T11RPM.2026059T182209.v2.0/HLS.S30.T11RPM.2026059T182209.v2.0.B02.tif
    Saved figure: HLS.S30.T11RPM.2026059T182209.v2.0.B02_comparison.png

Comparison complete - 1 files processed, 1 comparison figures saved


,file,product_type,pct_nonzero_diff,max_abs_diff_refl,pct_of_range,filesize_identical,figure_path
0,outputs/exp1-recompile/prod/S30/data/2026059/H...,SR,0.04266,0.0002,0.04128,True,/Users/vo25-mbp/hls-validation/hls_validation_...


In [135]:
df_results = pd.concat(df_diff_list, ignore_index=True)


---
## 7. Statistical Testing & Pass / Fail Decision

### Purpose
This section answers the critical engineering question:

> **Are the pixel-level differences between the reference (prod) and candidate (dev) containers statistically significant, or are they indistinguishable from random floating-point noise?**

The analysis runs two complementary, fully non-parametric tests **per band** (across all granules of that band type) and derives a per-band and overall **Pass / Fail** verdict from their combined result.

No user-defined thresholds or reference margins are required — the decision is driven entirely by the data.

---

### Tests

| # | Test | H₀ (null hypothesis) | What it detects | Reference |
|---|---|---|---|---|
| 1 | **Paired Wilcoxon signed-rank** | Median per-file mean difference = 0 | Consistent directional bias (systematic shift) | Wilcoxon (1945) |
| 2 | **Permutation test** | Observed mean difference arises by random chance under sign-flipping | Any non-zero mean shift, with no distributional assumption | Good (2000) |

Both tests are applied to the vector of **per-file mean differences** (one value per granule per band).  
Because the tests are non-parametric, they remain valid even when most values are exactly zero.

---

### How the Pass / Fail decision is derived

A band is awarded **PASS** if **both** of the following hold at the chosen significance level (α):

| Criterion | Condition | Meaning |
|---|---|---|
| Wilcoxon | p ≥ α | No statistically significant median bias across granules |
| Permutation | p ≥ α | Observed mean difference is not extreme under random sign-flipping |

A band **FAILS** if either test returns p < α, indicating that the observed differences are unlikely to be random noise and may reflect a real systematic change introduced by the new container.

The **overall section verdict** is:
- **PASS** — all bands pass both criteria
- **FAIL** — one or more bands fail at least one criterion

Results are printed as a per-band table (cells below) and saved to `reports/`.

In [148]:
from scipy import stats

# ── Load results ──────────────────────────────────────────────────────────────
# Option A: use df_results already in memory from Section 6 (run notebook top-down)
# Option B: load a previously saved CSV directly
LOAD_FROM_CSV = True    # set True to load from a saved file instead
CSV_PATH = '/Users/vo25-mbp/Documents/NASA-IMPACT/HLS/Container_testing/stats/prod_vs_dev_S30_20260402_1200_no_twin - prod_vs_dev_S30_20260402_1200_no_twin.csv'

if LOAD_FROM_CSV:
    df_stat_input = pd.read_csv(CSV_PATH)
else:
    df_stat_input = df_results.copy()

# ── Column name mapping ───────────────────────────────────────────────────────
# CSV uses:  filename_old_container, mean_diff (DN float), max_abs_diff (DN int)
# Notebook uses: filename_pre, mean_diff_refl, max_abs_diff_refl
# Normalise to a common set of working column names.
col_map = {
    'filename_old_container': 'filename_pre',
    'filename_new_container': 'filename_post',
    'max_old_container'     : 'max_pre',
    'max_new_container'     : 'max_post',
    'mean_old_container'    : 'mean_pre',
    'mean_new_container'    : 'mean_post',
    'filesize_old_container': 'filesize_pre',
    'filesize_new_container': 'filesize_post',
}
df_stat_input = df_stat_input.rename(columns={k: v for k, v in col_map.items() if k in df_stat_input.columns})

# ── Derive band and product_type from filename ────────────────────────────────
def extract_band(filepath):
    """Extract band suffix, e.g. 'B01', 'NDVI', 'Fmask'."""
    fname = str(filepath).split('/')[-1]   # HLS.S30.T11RPM....B01.tif
    parts = fname.split('.')
    return parts[-2] if len(parts) >= 2 else fname

def extract_product_type(filepath):
    """Return 'VI' if VI product path, else 'SR'."""
    return 'VI' if '_VI' in str(filepath) else 'SR'

df_stat_input['band']         = df_stat_input['filename_pre'].apply(extract_band)
df_stat_input['product_type'] = df_stat_input['filename_pre'].apply(extract_product_type)

# ── Convert DN columns to physical reflectance / index units ─────────────────
# mean_diff in CSV is in raw DN (fractional float); max_abs_diff is integer DN.
# Physical value = DN * HLS_SCALE_FACTOR (0.0001)
df_stat_input['mean_diff_refl']    = df_stat_input['mean_diff']    * HLS_SCALE_FACTOR
df_stat_input['max_abs_diff_refl'] = df_stat_input['max_abs_diff'] * HLS_SCALE_FACTOR
df_stat_input['filesize_identical']= df_stat_input['filesize_pre'] == df_stat_input['filesize_post']

ALPHA = 0.05

# ── Per-band statistical tests ────────────────────────────────────────────────
stat_rows = []

for band, grp in df_stat_input.groupby('band'):
    n_files    = len(grp)
    pt         = grp['product_type'].iloc[0]
    mean_diffs = grp['mean_diff_refl'].values     # per-file mean diff in reflectance
    max_diffs  = grp['max_abs_diff_refl'].values  # per-file max |diff| in reflectance

    # ── 1. Wilcoxon signed-rank test ──────────────────────────────────────────
    # H0: median(mean_diff) == 0 — no systematic bias between containers
    # Non-parametric; does not assume normality.
    if n_files >= 2 and not np.all(mean_diffs == 0):
        _, p_wilcox = stats.wilcoxon(mean_diffs, alternative='two-sided')
    elif np.all(mean_diffs == 0):
        p_wilcox = 1.0
    else:
        p_wilcox = float('nan')

    sig_wilcox = bool(p_wilcox < ALPHA) if not np.isnan(p_wilcox) else None

    # ── 3. Permutation test ───────────────────────────────────────────────────
    # H0: the observed mean difference arises by chance under random sign-flipping.
    # Fully data-driven — no reference margin, no distributional assumption.
    if n_files >= 2 and not np.all(mean_diffs == 0):
        _perm_res = stats.permutation_test(
            (mean_diffs,),
            lambda x: np.mean(x),
            permutation_type='samples',   # randomly flips signs
            n_resamples=9_999,
            alternative='two-sided',
            random_state=0,
        )
        p_perm = float(_perm_res.pvalue)
    elif np.all(mean_diffs == 0):
        p_perm = 1.0
    else:
        p_perm = float('nan')

    stat_rows.append({
        'band'              : band,
        'product_type'      : pt,
        'n_files'           : n_files,
        'n_files_with_diff' : int((grp['pct_nonzero_diff'] > 0).sum()),
        'mean_diff_refl'    : round(float(np.mean(mean_diffs)),  6),
        'max_abs_diff_refl' : round(float(np.max(max_diffs)),    6),
        'pct_pixels_changed': round(float(grp['pct_nonzero_diff'].max()), 5),
        'wilcoxon_p'        : round(p_wilcox, 4) if not np.isnan(p_wilcox) else float('nan'),
        'significant_bias'  : sig_wilcox,
        'perm_p'            : round(p_perm, 4) if not np.isnan(p_perm) else float('nan'),
        'perm_significant'  : bool(p_perm < ALPHA) if not np.isnan(p_perm) else None,
    })

df_stats = pd.DataFrame(stat_rows).sort_values(['product_type', 'band'])

print("=" * 90)
print("  STATISTICAL TEST RESULTS — PER BAND")
print(f"  Input : {'CSV: ' + CSV_PATH if LOAD_FROM_CSV else 'df_results (in-memory)'}")
print(f"  Scale : DN x {HLS_SCALE_FACTOR} = reflectance/index units")
print(f"  Wilcoxon alpha = {ALPHA}  |  TOST margin = {EQUIVALENCE_MARGIN_REFL} reflectance units")
print("=" * 90)
print(df_stats.to_string(index=False))
print("=" * 90)


  STATISTICAL TEST RESULTS — PER BAND
  Input : CSV: /Users/vo25-mbp/Documents/NASA-IMPACT/HLS/Container_testing/stats/prod_vs_dev_S30_20260402_1200_no_twin - prod_vs_dev_S30_20260402_1200_no_twin.csv
  Scale : DN x 0.0001 = reflectance/index units
  Wilcoxon alpha = 0.05  |  TOST margin = None reflectance units
 band product_type  n_files  n_files_with_diff  mean_diff_refl  max_abs_diff_refl  pct_pixels_changed  wilcoxon_p  significant_bias  perm_p  perm_significant
  B01           SR       15                  1             0.0             0.0001             0.00003      1.0000             False  1.0000             False
  B02           SR       15                  1             0.0             0.0002             0.04266      0.3173             False  0.9982             False
  B03           SR       15                  1             0.0             0.0002             0.00001      1.0000             False  1.0000             False
  B04           SR       15                  1        

In [151]:
# ── Section 7 Pass / Fail — purely based on statistical tests ────────────────
# A band PASSES if BOTH criteria hold:
#   (1) Wilcoxon p >= ALPHA          → no significant median bias
#   (2) Permutation p >= ALPHA       → mean difference not extreme by chance

PASS_ICON = 'PASS'
FAIL_ICON = 'FAIL'

WIDTH = 80
sep   = '=' * WIDTH

rows = []
for _, r in df_stats.iterrows():
    w_pass  = (r['wilcoxon_p'] >= ALPHA)  if not pd.isna(r['wilcoxon_p'])  else True
    p_pass  = (r['perm_p']     >= ALPHA)  if not pd.isna(r['perm_p'])      else True

    band_pass = w_pass and p_pass

    rows.append({
        'Band'         : r['band'],
        'Type'         : r['product_type'],
        'Wilcoxon p'  : f"{r['wilcoxon_p']:.4f}" if not pd.isna(r['wilcoxon_p']) else 'n/a',
        'Wilcoxon'    : PASS_ICON if w_pass else FAIL_ICON,
        'Perm p'      : f"{r['perm_p']:.4f}"     if not pd.isna(r['perm_p'])     else 'n/a',
        'Perm test'   : PASS_ICON if p_pass else FAIL_ICON,
        'Band verdict': PASS_ICON if band_pass else FAIL_ICON,
    })

df_pf = pd.DataFrame(rows)

# ── Print table ───────────────────────────────────────────────────────────────
print(sep)
print('  SECTION 7 — STATISTICAL PASS / FAIL')
print(f'  Criteria: Wilcoxon p >= {ALPHA}  AND  Permutation p >= {ALPHA}  AND  |d| < 0.2')
print(sep)
print(df_pf.to_string(index=False))
print(sep)

# ── Overall Section 9 verdict ─────────────────────────────────────────────────
n_fail_bands = int((df_pf['Band verdict'] == FAIL_ICON).sum())
n_total      = len(df_pf)

if n_fail_bands == 0:
    s9_verdict  = PASS_ICON
    s9_note     = (f'All {n_total} bands pass both statistical criteria. '
                   f'No statistically significant bias detected.')
else:
    s9_verdict  = FAIL_ICON
    fail_bands  = df_pf[df_pf['Band verdict'] == FAIL_ICON]['Band'].tolist()
    s9_note     = (f'{n_fail_bands}/{n_total} band(s) fail at least one statistical criterion: '
                   f'{", ".join(fail_bands)}. Review df_stats for details.')

box_w = WIDTH - 2
pad   = (box_w - len(s9_verdict)) // 2
print()
print('|' + ' ' * box_w + '|')
print('|' + ' ' * pad + s9_verdict + ' ' * (box_w - pad - len(s9_verdict)) + '|')
print('|' + ' ' * box_w + '|')
print(sep)
print(f'  {s9_note}')
print(sep)

# ── Save ──────────────────────────────────────────────────────────────────────
_ts    = datetime.now().strftime('%Y%m%d_%H%M%S')
_fpath = os.path.join(REPORT_DIR, f'statistical_passfail_{_ts}.csv')
df_pf.to_csv(_fpath, index=False)
print(f'\nSaved to: {_fpath}')


  SECTION 9 — STATISTICAL PASS / FAIL
  Criteria: Wilcoxon p >= 0.05  AND  Permutation p >= 0.05  AND  |d| < 0.2
 Band Type Wilcoxon p Wilcoxon Perm p Perm test Band verdict
  B01   SR     1.0000     PASS 1.0000      PASS         PASS
  B02   SR     0.3173     PASS 0.9982      PASS         PASS
  B03   SR     1.0000     PASS 1.0000      PASS         PASS
  B04   SR     1.0000     PASS 1.0000      PASS         PASS
  B05   SR     1.0000     PASS 1.0000      PASS         PASS
  B06   SR     1.0000     PASS 1.0000      PASS         PASS
  B07   SR     1.0000     PASS 1.0000      PASS         PASS
  B08   SR     1.0000     PASS 1.0000      PASS         PASS
  B09   SR     1.0000     PASS 1.0000      PASS         PASS
  B10   SR     1.0000     PASS 1.0000      PASS         PASS
  B11   SR     1.0000     PASS 1.0000      PASS         PASS
  B12   SR     1.0000     PASS 1.0000      PASS         PASS
  B8A   SR     1.0000     PASS 1.0000      PASS         PASS
Fmask   SR     1.0000     PASS 1.

In [152]:
# ── Statistical conclusion (dynamic — adapts to any result) ──────────────────

n_bands_total   = len(df_stats)
n_sig           = int(df_stats['significant_bias'].sum())
n_any_diff      = int((df_stats['n_files_with_diff'] > 0).sum())
n_zero_diff     = int((df_stats['n_files_with_diff'] == 0).sum())
sig_bands       = df_stats[df_stats['significant_bias'] == True]['band'].tolist()
max_diff_row    = df_stats.loc[df_stats['max_abs_diff_refl'].idxmax()]
max_pct_row     = df_stats.loc[df_stats['pct_pixels_changed'].idxmax()]
all_mean_zero   = (df_stats['mean_diff_refl'].abs() < 1e-9).all()
n_granules      = int(df_stats['n_files'].iloc[0])
mean_diff_global = float(df_stats['mean_diff_refl'].mean())

# ── Determine overall significance tier ──────────────────────────────────────
# Tier 0 - no bias detected
# Tier 2 - bias in a minority of bands (≤25%)
# Tier 3 - bias in majority of bands
pct_sig = n_sig / n_bands_total * 100
if n_sig == 0:
    tier = 0
    sig_label = 'NOT statistically significant'
elif pct_sig <= 25:
    tier = 2
    sig_label = f'MARGINALLY significant ({n_sig}/{n_bands_total} bands, p < {ALPHA})'
else:
    tier = 3
    sig_label = f'STATISTICALLY SIGNIFICANT ({n_sig}/{n_bands_total} bands, p < {ALPHA})'

# ── Bias section text ─────────────────────────────────────────────────────────
if n_sig == 0:
    bias_line = (f'   0/{n_bands_total} bands show a statistically significant bias (p < {ALPHA}).'
                 + ('\n   All per-band mean differences = 0.0 reflectance -- no systematic offset.'
                    if all_mean_zero else
                    f'\n   Global mean difference across bands: {mean_diff_global:.6f} reflectance units.'))
else:
    sig_band_str = ', '.join(sig_bands)
    bias_line = (f'   {n_sig}/{n_bands_total} bands show a statistically significant bias (p < {ALPHA}):\n'
                 f'   Affected bands: {sig_band_str}\n'
                 f'   Global mean diff: {mean_diff_global:.6f} reflectance units.')



# ── Interpretation block ─────────────────────────────────────────────────────
if tier == 0:
    interp = ('   Differences are isolated to a subset of granules and are consistent\n'
              '   with floating-point arithmetic variation between compiler environments.\n'
              '   They are NOT scientifically meaningful.')

elif tier == 2:
    interp = ('   A subset of bands shows statistically significant bias. Differences\n'
              '   are small in magnitude but consistent enough to be detected. Investigate\n'
              '   whether the affected bands share a common processing step.')
else:
    interp = ('   Statistically significant bias was detected in a majority of bands.\n'
              '   The magnitude and consistency of differences suggest a real processing\n'
              '   change. Manual inspection of the affected granules and bands is required.')

ref_compare = (f'   Max deviation ({max_diff_row["max_abs_diff_refl"]:.4f} reflectance) vs reference benchmarks:\n'
               f'     - HLS cross-sensor consistency    : ~0.042 reflectance (4.2%)\n'
               f'     - Atmospheric correction uncertainty: ~0.02-0.05 reflectance')

lines = [
    '=' * 80,
    '  CONCLUSION -- STATISTICAL SIGNIFICANCE SUMMARY',
    '=' * 80,
    '',
    f'OVERALL VERDICT: Differences are {sig_label} (alpha = {ALPHA}).',
    '',
    f'Dataset: {n_bands_total} bands/products, {n_granules} granules per band.',
    '',
    '1. SYSTEMATIC BIAS (Wilcoxon signed-rank test)',
    bias_line,
    '',

    '3. MAGNITUDE OF DIFFERENCES',
    f'   Largest max |diff|    : {max_diff_row["band"]} -- '
    f'{max_diff_row["max_abs_diff_refl"]:.4f} reflectance units'
    f' (= {int(max_diff_row["max_abs_diff_refl"] / HLS_SCALE_FACTOR)} DN)',
    f'   Largest % pix changed : {max_pct_row["band"]} -- {max_pct_row["pct_pixels_changed"]:.5f}% of pixels',
    f'   Bit-for-bit identical : {n_zero_diff}/{n_bands_total} bands across all granules',
    f'   Bands with any diff   : {n_any_diff}/{n_bands_total}',
    '',
    '4. INTERPRETATION',
    interp,
    ref_compare,
    '=' * 80,
]

print('\n'.join(lines))

# ── Save conclusion ───────────────────────────────────────────────────────────
_ts    = datetime.now().strftime('%Y%m%d_%H%M%S')
_fpath = os.path.join(REPORT_DIR, f'conclusion_{_ts}.txt')
with open(_fpath, 'w') as _f:
    _f.write('\n'.join(lines))
print(f'\nConclusion saved to: {_fpath}')


  CONCLUSION -- STATISTICAL SIGNIFICANCE SUMMARY

OVERALL VERDICT: Differences are NOT statistically significant (alpha = 0.05).

Dataset: 27 bands/products, 15 granules per band.

1. SYSTEMATIC BIAS (Wilcoxon signed-rank test)
   0/27 bands show a statistically significant bias (p < 0.05).
   All per-band mean differences = 0.0 reflectance -- no systematic offset.

3. MAGNITUDE OF DIFFERENCES
   Largest max |diff|    : NDVI -- 0.0033 reflectance units (= 33 DN)
   Largest % pix changed : B02 -- 0.04266% of pixels
   Bit-for-bit identical : 10/27 bands across all granules
   Bands with any diff   : 17/27

4. INTERPRETATION
   Differences are isolated to a subset of granules and are consistent
   with floating-point arithmetic variation between compiler environments.
   They are NOT scientifically meaningful.
   Max deviation (0.0033 reflectance) vs reference benchmarks:
     - HLS cross-sensor consistency    : ~0.042 reflectance (4.2%)
     - Atmospheric correction uncertainty: ~0.02-0